<a href="https://colab.research.google.com/github/xyt556/I-GUIDE-GeoAI-Education/blob/main/notebooks/03-training-data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 创建训练数据

## 介绍

任何 GeoAI 模型的性能都取决于其训练数据的质量和数量。数据准备通常占用项目总时间的 60-80%，而在此投入的精力直接决定了结果的质量。

本教程涵盖了为地理空间深度学习创建训练数据集的端到端工作流程。您将学习如何将矢量标签转换为栅格掩模，使用 `geoai` 包生成图像切片，处理单图像和批量工作流程，以及可视化训练瓦片。

## 学习目标

完成本教程后，您将能够：

- 描述从原始图像到模型就绪数据集的完整训练数据管道
- 使用 `geoai` 将矢量注释转换为栅格掩模
- 使用瓦片和平铺策略从大型卫星场景生成图像切片
- 从单图像以及图像和掩模的批量文件夹创建训练瓦片
- 使用三种不同的输入模式将图像与矢量注释配对
- 可视化训练数据以验证标签质量和对齐
- 将数据集组织成具有空间分离的训练、验证和测试拆分

## 训练数据管道

为 GeoAI 创建训练数据将原始地理空间数据转换为深度学习框架所需的结构化格式。该管道遵循一致的模式：

1. **获取图像和标签**：获取卫星或航空图像以及相应的注释。
2. **将矢量标签转换为栅格掩模**：将矢量注释栅格化到与图像分辨率和坐标系匹配的网格上。
3. **对图像进行瓦片处理**：将大型栅格场景分割成固定大小的图像切片。
4. **生成配对标签**：为每个图像切片生成相应的标签瓦片。
5. **可视化和验证**：在训练前检查生成的瓦片以验证标签对齐和质量。
6. **拆分为子集**：将数据集拆分为训练、验证和测试集，注意保持空间分离。

In [ ]:
# %pip install -U "geoai-py[extra]" lxml

In [ ]:
import os
import sys


def configure_proj_data(proj_dir: str) -> str:
    """Force PROJ's data directory through both env vars and library APIs.

    Setting ``PROJ_LIB``/``PROJ_DATA`` only takes effect if done before
    rasterio's C extension initializes PROJ. This helper additionally calls
    pyproj's ``set_data_dir`` as a belt-and-suspenders measure for builds
    that ignore the environment variables. It must still run before any CRS
    lookup is triggered (i.e. before ``leafmap`` / ``rio_tiler`` import).

    Args:
        proj_dir: Directory containing ``proj.db``.

    Returns:
        The configured PROJ data directory.

    Raises:
        AssertionError: If rasterio/leafmap were imported before this ran.
        FileNotFoundError: If ``proj.db`` is not present in ``proj_dir``.
    """
    assert "rasterio" not in sys.modules, "Restart the kernel; rasterio already loaded."
    if not os.path.exists(os.path.join(proj_dir, "proj.db")):
        raise FileNotFoundError(f"proj.db not in {proj_dir}")

    os.environ["PROJ_LIB"] = proj_dir
    os.environ["PROJ_DATA"] = proj_dir

    import pyproj

    pyproj.datadir.set_data_dir(proj_dir)
    return proj_dir


configure_proj_data(
    "/cvmfs/iguide.purdue.edu/software/conda/geoai-edu/lib/python3.12/site-packages/rasterio/proj_data"
)

## 从单张图像生成图像切片

卫星和航空影像场景太大，无法直接输入神经网络。切片（或称“分块”）将这些大型场景切割成易于处理的碎片，通常为 256 x 256 或 512 x 512 像素。

### 下载示例数据

我们首先下载一个示例 NAIP 图像及其对应的建筑物轮廓标注。

## 从单张图像生成图像切片

卫星和航空影像场景太大，无法直接输入神经网络。切片（或称“分块”）将这些大型场景切割成易于处理的碎片，通常为 256 x 256 或 512 x 512 像素。

### 下载示例数据

我们首先下载一个示例 NAIP 图像及其对应的建筑物轮廓标注。

In [ ]:
import geoai

In [ ]:
raster_url = "https://data.source.coop/opengeos/geoai/naip-train.tif"
vector_url = "https://data.source.coop/opengeos/geoai/naip-train-buildings.geojson"
raster_path = geoai.download_file(raster_url)
vector_path = geoai.download_file(vector_url)

### 预览数据

在生成切片之前，目视检查源图像和标注，以发现标签偏移或图像伪影等问题。

### 预览数据

在生成切片之前，目视检查源图像和标注，以发现标签偏移或图像伪影等问题。

In [ ]:
geoai.view_image(raster_path, figsize=(18, 10))

In [ ]:
geoai.view_vector(vector_path, raster_path=raster_path, figsize=(18, 10))

In [ ]:
geoai.view_vector_interactive(vector_path, tiles=raster_path)

### 将矢量转换为栅格

语义分割模型需要像素级标签。`vector_to_raster()` 函数将矢量多边形栅格化到与参考栅格的分辨率、范围和坐标系（CRS）相匹配的网格上，生成二值掩模（1 代表建筑物，0 代表背景）。

### 将矢量转换为栅格

语义分割模型需要像素级标签。`vector_to_raster()` 函数将矢量多边形栅格化到与参考栅格的分辨率、范围和坐标系（CRS）相匹配的网格上，生成二值掩模（1 代表建筑物，0 代表背景）。

In [ ]:
output_path = vector_path.replace(".geojson", ".tif")
geoai.vector_to_raster(vector_path, output_path, reference_raster=raster_path)

In [ ]:
geoai.view_image(output_path, figsize=(18, 10))

### 切片参数

切片生成的关键参数包括：

- **切片大小 (Tile size)**：每个切片的像素宽高。应与模型预期的输入尺寸匹配。
- **步长 (Stride)**：连续切片之间的步长。步长等于切片大小时无重叠；步长较小时则有重叠。

### 生成切片

`export_geotiff_tiles()` 函数处理完整的切片生成流程：对影像进行切片、将矢量标签栅格化为掩模，并保存成对的图像-掩模切片。

### 切片参数

切片生成的关键参数包括：

- **切片大小 (Tile size)**：每个切片的像素宽高（例如 256x256 或 512x512）。这应与模型预期的输入尺寸匹配。
- **步长 (Stride)**：连续切片之间的步长。步长等于切片大小时无重叠；步长较小时则有重叠。

重叠切片可确保位于切片边界的对象在至少一个切片中完整出现。常用重叠比例：

| 重叠率 | 步长 (256px 切片) | 步长 (512px 切片) | 适用场景 |
| ------- | ------------------------ | ------------------------ | ------------------------------------------- |
| 0%      | 256                      | 512                      | 简单分类，大面积均匀区域 |
| 25%     | 192                      | 384                      | 通用目的，中等程度的边界对象 |
| 50%     | 128                      | 256                      | 密集目标检测，建筑物轮廓 |
| 75%     | 64                       | 128                      | 最大覆盖，小目标检测 |

较高的重叠率会产生更多切片并增加存储需求。请根据对象相对于切片尺寸的大小以及计算预算选择重叠率。

### 生成切片

`export_geotiff_tiles()` 函数处理完整的切片生成流程：对影像进行切片、将矢量标签栅格化为掩模，并保存成对的图像-掩模切片。

In [ ]:
tiles = geoai.export_geotiff_tiles(
    in_raster=raster_path,
    out_folder="output",
    in_class_data=vector_path,
    tile_size=512,
    stride=384,
    buffer_radius=0,
    create_overview=True,
    quiet=True,
)

### 预览图像切片

生成切片后，检查概览图和单个图像-掩模对，以验证标签是否正确对齐。

In [ ]:
geoai.view_image("output/overview.png", figsize=(18, 10))

使用 `display_training_tiles()` 函数可视化生成的切片。

In [ ]:
fig = geoai.display_training_tiles(output_dir="output", num_tiles=4, figsize=(18, 10))

## 批量处理多张图像

在实际项目中，训练数据往往跨越多个卫星场景。`export_geotiff_tiles_batch()` 函数支持批量处理，并提供三种配对模式：

1. **覆盖所有图像的单个矢量文件** - 当你有一个大型标注文件时最有效。
2. **按排序顺序匹配多个矢量文件** - 适用于顺序数据集。
3. **按文件名匹配多个矢量文件** - 适用于文件名对应的配对数据集。

## 批量处理多张图像

在实际项目中，训练数据往往跨越多个卫星场景。`export_geotiff_tiles_batch()` 函数支持批量处理，并提供三种配对模式：

1. **覆盖所有图像的单个矢量文件** - 当你有一个大型标注文件时最有效。
2. **按排序顺序匹配多个矢量文件** - 适用于顺序数据集。
3. **按文件名匹配多个矢量文件** - 适用于文件名对应的配对数据集。

### 下载批量示例数据

示例数据集包含两个 NAIP 图像切片以及两种格式的建筑物标注：一个覆盖所有切片的单个 GeoJSON 文件，以及每个切片的独立 GeoJSON 文件。

In [ ]:
import os

In [ ]:
from pathlib import Path
import shutil

data_dir = Path.cwd() / "data"

if data_dir.exists():
    shutil.rmtree(data_dir)

In [ ]:
url = "https://data.source.coop/opengeos/geoai/naip-rgb-train-tiles.zip"
data_dir = geoai.download_file(url)

### 探索示例数据

让我们列出示例数据目录中的文件，看看包含哪些内容：

In [ ]:
print("Images:")
for f in sorted(os.listdir(f"{data_dir}/images")):
    print(f"  - {f}")

print("\nAnnotations (single file):")
for f in sorted(os.listdir(f"{data_dir}/masks1")):
    print(f"  - {f}")

print("\nAnnotations (multiple files):")
for f in sorted(os.listdir(f"{data_dir}/masks2")):
    print(f"  - {f}")

### 可视化图像和标注

`display_image_with_vector()` 函数在源图像上叠加矢量标注，以便快速目视检查标签对齐情况。

In [ ]:
image_path = f"{data_dir}/images/naip_rgb_train_tile1.tif"
mask_path = f"{data_dir}/masks2/naip_rgb_train_tile1.geojson"

fig, axes, info = geoai.display_image_with_vector(image_path, mask_path)
print(f"Number of buildings: {info['num_features']}")

### 方法 1：覆盖所有图像的单个矢量文件

此方法使用一个覆盖多个图像切片的标注文件。函数会根据每个图像的边界空间过滤要素。

In [ ]:
stats = geoai.export_geotiff_tiles_batch(
    images_folder=f"{data_dir}/images",
    masks_file=f"{data_dir}/masks1/naip_train_buildings.geojson",
    output_folder="output/method1_single_mask",
    tile_size=256,
    stride=128,
    class_value_field="class",
    skip_empty_tiles=True,
    quiet=False,
)

print(f"\n{'='*60}")
print("Results:")
print(f"  Images processed: {stats['processed_pairs']}")
print(f"  Total tiles generated: {stats['total_tiles']}")
print(f"  Tiles with features: {stats['tiles_with_features']}")
print(
    f"  Feature percentage: {stats['tiles_with_features']/stats['total_tiles']*100:.1f}%"
)

### 方法 2：按排序顺序匹配多个矢量文件

此方法按字母顺序对图像和掩模进行配对。仅当您确信排序后的文件列表能够正确对应时才使用此模式。

In [ ]:
stats = geoai.export_geotiff_tiles_batch(
    images_folder=f"{data_dir}/images",
    masks_folder=f"{data_dir}/masks2",
    output_folder="output/method2_sorted_order",
    tile_size=256,
    stride=128,
    class_value_field="class",
    skip_empty_tiles=True,
    match_by_name=False,
)

print(f"\n{'='*60}")
print("Results:")
print(f"  Images processed: {stats['processed_pairs']}")
print(f"  Total tiles generated: {stats['total_tiles']}")
print(f"  Tiles with features: {stats['tiles_with_features']}")

### 方法 3：按文件名匹配多个矢量文件

此方法通过匹配基本文件名（例如 `tile1.tif` 与 `tile1.geojson`）来配对图像和掩模。当文件共享相同基本名称时，这是最安全的选择。

In [ ]:
stats = geoai.export_geotiff_tiles_batch(
    images_folder=f"{data_dir}/images",
    masks_folder=f"{data_dir}/masks2",
    output_folder="output/method3_matched_name",
    tile_size=256,
    stride=128,
    class_value_field="class",
    skip_empty_tiles=True,
    match_by_name=True,
)

print(f"\n{'='*60}")
print("Results:")
print(f"  Images processed: {stats['processed_pairs']}")
print(f"  Total tiles generated: {stats['total_tiles']}")
print(f"  Tiles with features: {stats['tiles_with_features']}")

### 可视化生成的切片

`display_training_tiles()` 函数显示输出目录中图像-掩模对的网格。

In [ ]:
output_dir = "output/method1_single_mask"
fig = geoai.display_training_tiles(output_dir, num_tiles=4, figsize=(18, 10))

### 高级用法：自定义参数

`export_geotiff_tiles_batch()` 函数支持许多自定义参数，包括切片大小、步长、缓冲区半径和空切片过滤。

In [ ]:
stats = geoai.export_geotiff_tiles_batch(
    images_folder=f"{data_dir}/images",
    masks_file=f"{data_dir}/masks1/naip_train_buildings.geojson",
    output_folder="output/advanced_example",
    tile_size=512,
    stride=256,
    class_value_field="class",
    buffer_radius=0.5,
    skip_empty_tiles=True,
    all_touched=True,
    max_tiles=10,
    quiet=False,
)

print(f"\nGenerated {stats['total_tiles']} tiles with 50% overlap")
print(f"Output structure:")
print(f"  - output/advanced_example/images/  (image tiles)")
print(f"  - output/advanced_example/masks/   (mask tiles)")

## 使用栅格掩模进行批量处理

当您的标注已经是栅格化格式（例如土地覆盖图）时，可以直接传递栅格掩模文件夹而不是矢量文件。

In [ ]:
url = "https://data.source.coop/opengeos/geoai/landcover-sample-data.zip"
data_dir2 = geoai.download_file(url)

In [ ]:
images_dir = f"{data_dir2}/images"
masks_dir = f"{data_dir2}/masks"
tiles_dir = f"{data_dir2}/tiles"

In [ ]:
result = geoai.export_geotiff_tiles_batch(
    images_folder=images_dir,
    masks_folder=masks_dir,
    output_folder=tiles_dir,
    tile_size=512,
    stride=384,
    quiet=True,
)

## 总结与要点

1. **训练数据质量决定模型质量**：在训练前投入时间进行仔细的标注和验证。
2. **切片是必不可少的**：用于处理大型卫星场景；使用重叠策略处理边缘对象。
3. **始终进行可视化**：在训练前使用 `display_training_tiles()` 检查数据。
4. **空间隔离**：在划分训练/验证/测试集时，确保地理上的空间分离以防止数据泄露。

生成的切片存储在 `images/` 和 `masks/` 子文件夹中，每个图像切片都有对应的掩模切片，可随时用于训练。

## 标签质量注意事项

高质量的标签对模型性能至关重要。需要注意的常见问题包括：

- **标注不完整**：训练数据中缺失的对象会教导模型忽略这些对象。确保每个切片中目标类的所有实例都已标记。
- **空间对齐偏移**：与图像未精确对齐的标签会引入噪声。始终使用 `display_image_with_vector()` 验证矢量标注是否正确叠加在栅格上。
- **类别不平衡**：当某一类别在数据集中占主导地位时，模型可能会只学会预测多数类。考虑使用过采样、类别权重或 Focal Loss。
- **边缘效应**：切片边界处的对象可能被部分标记。之前描述的重叠策略可以减轻此问题。`skip_empty_tiles` 参数也有助于排除没有要素的切片。
- **缓冲半径**：在矢量要素周围添加小的缓冲区（使用 `buffer_radius` 参数）可以弥补影像与标注之间的轻微对齐误差。

## 数据集组织

### 训练/验证/测试集划分

划分数据集需要**空间隔离**。相邻的切片高度相关，如果训练和验证切片来自相邻位置，模型可能会获得虚高的验证分数。

为避免数据泄露，应在场景或区域级别划分数据：

- **基于区域的划分**：将整个地理区域分配给不同的划分。例如，使用一个城市的影像进行训练，另一个城市进行验证。
- **基于场景的划分**：如果处理多个卫星场景，将整个场景分配给不同的划分。
- **空间缓冲区**：在单个场景内划分时，在训练和验证区域之间留出缓冲区（若干个切片宽度）。

常用的划分比例为 70/15/15 或 80/10/10。

### 目录结构

大多数深度学习框架期望数据以特定的目录布局组织。`geoai` 包生成以下结构：

**分割数据集**（图像-掩模对）：

```
dataset/
    images/
        tile_000000.tif
        ...
    masks/
        tile_000000.tif
        ...
```

## 总结

本教程介绍了将原始地理空间数据转换为模型就绪训练数据集的单图像和批量工作流程。对于批量处理，`export_geotiff_tiles_batch()` 提供了三种灵活的配对方法：

| 方法 | 使用场景 | 参数 |
| ------------------------- | --------------------------------------- | ------------------------------------------- |
| 单个矢量文件 | 一个标注文件覆盖所有图像 | `masks_file="path/to/file.geojson"` |
| 多个文件（按顺序） | 按排序顺序排列的成对文件 | `masks_folder="path/", match_by_name=False` |
| 多个文件（按名称） | 具有匹配名称的成对文件 | `masks_folder="path/", match_by_name=True` |

## 核心要点

1. **训练数据质量决定模型质量** - 在训练前投入时间进行仔细标注和验证。
2. **切片是必不可少的** - 用于处理大型卫星场景；使用重叠处理边界对象。
3. **始终进行可视化** - 使用 `display_image_with_vector()` 和 `display_training_tiles()` 检查数据。
4. **空间隔离** - 在划分数据集时防止空间自相关导致的数据泄露。